# 03 · Modern CNNs: ResNet and EfficientNet

**Covers (Week 3 Key Concepts, "Why Study Modern CNNs?"):** ResNet (2015), EfficientNet (2019), plus MobileNet for "lightweight models for phones".
**Questions answered here:** Easy 10–14 · Medium 5–9 · Hard 3–4, 7.

**The story so far:** VGG showed that deeper means more accurate. So why not 100 layers? People tried, and **very deep plain networks got *worse***, even on their *training* data. Two problems stood in the way:
1. **Depth:** gradients vanish, so the early layers stop learning. **ResNet** fixed this.
2. **Efficiency:** bigger networks are slow and wasteful if you grow them the wrong way. **EfficientNet** fixed this.

In [1]:
%matplotlib inline
import time
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

torch.manual_seed(0)
digits = load_digits()
X = torch.tensor(digits.images, dtype=torch.float32)[:, None] / 16.0
X = F.interpolate(X, size=(16, 16), mode="bilinear", align_corners=False)
y = torch.tensor(digits.target)
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)
print("data:", tuple(Xtr.shape), tuple(Xte.shape))

data: (1257, 1, 16, 16) (540, 1, 16, 16)


## 1 · ResNet (2015): the residual connection

### The idea in one picture

```
Plain block:          x ──► [conv → ReLU → conv] ──► output = H(x)

Residual block:       x ──► [conv → ReLU → conv] ──► F(x) ──(+)──► output = F(x) + x
                      │                                  ▲
                      └──────────── skip connection ─────┘   (the input jumps over the layers)
```

- A **plain** block must learn the whole desired output **H(x)** (a "direct mapping").
- A **residual** block only learns the **difference** (the *residual*) **F(x) = H(x) − x**, i.e. what to *change* about its input. The input is then added back.

**Analogy:** instead of redrawing a picture from scratch at every layer, each layer only makes **corrections** to the picture it received. If a layer has nothing useful to add, it can learn F(x) ≈ 0 and simply pass the input through unchanged. That's very easy. For a plain block, learning to "do nothing" (H(x) = x) is surprisingly hard.

### Why it fixes vanishing gradients
For output = F(x) + x, the gradient flowing backwards is:

  **d(output)/dx = F′(x) + 1**

That **"+1"** is a direct highway for the gradient. Even if F′(x) is tiny, the signal still passes back through the skip connection unchanged. In a plain network the gradient is a **product** of many F′ terms, and the product of many small numbers vanishes.

Let's build both kinds of block and **prove it**.

In [2]:
class Block(nn.Module):
    def __init__(self, channels, residual):
        super().__init__()
        self.conv1 = nn.Conv2d(channels, channels, 3, padding=1)
        self.conv2 = nn.Conv2d(channels, channels, 3, padding=1)
        self.residual = residual

    def forward(self, x):
        out = self.conv2(F.relu(self.conv1(x)))      # F(x)
        if self.residual:
            out = out + x                            # the skip connection: F(x) + x
        return F.relu(out)

class Net(nn.Module):
    def __init__(self, n_blocks, residual, channels=32):
        super().__init__()
        self.stem = nn.Conv2d(1, channels, 3, padding=1)
        self.blocks = nn.Sequential(*[Block(channels, residual) for _ in range(n_blocks)])
        self.head = nn.Linear(channels, 10)

    def forward(self, x):
        x = F.relu(self.stem(x))
        x = self.blocks(x)
        return self.head(x.mean(dim=(2, 3)))         # global average pooling -> classifier

def first_layer_gradient(model):
    model.zero_grad()
    F.cross_entropy(model(Xtr[:128]), ytr[:128]).backward()
    return model.stem.weight.grad.norm().item()     # how big is the learning signal reaching layer 1?

print(f"{'layers':>6} | {'plain net: gradient at layer 1':>32} | {'residual net: gradient at layer 1':>34}")
for n_blocks in [2, 10, 25]:
    torch.manual_seed(0); g_plain = first_layer_gradient(Net(n_blocks, residual=False))
    torch.manual_seed(0); g_res = first_layer_gradient(Net(n_blocks, residual=True))
    print(f"{2 * n_blocks + 2:>6} | {g_plain:>32.2e} | {g_res:>34.2e}")

layers |   plain net: gradient at layer 1 |  residual net: gradient at layer 1
     6 |                         3.00e-04 |                           1.81e-02


    22 |                         1.92e-10 |                           1.00e-02


    52 |                         0.00e+00 |                           2.04e-02


Read the left column: the deeper the plain network, the smaller the learning signal that reaches the first layer. At 22 layers it's around 10⁻¹⁰, and at 52 layers it's **exactly zero**: the first layer gets no information at all about how to improve. The residual network keeps a healthy signal at every depth.

### Does it actually change training? (≈ 3 minutes on a laptop CPU)

In [3]:
def train_eval(model, epochs=15, lr=3e-3, batch=64):
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    for _ in range(epochs):
        model.train()
        perm = torch.randperm(len(Xtr))
        for i in range(0, len(Xtr), batch):
            idx = perm[i:i + batch]
            loss = F.cross_entropy(model(Xtr[idx]), ytr[idx])
            opt.zero_grad(); loss.backward(); opt.step()
    model.eval()
    with torch.no_grad():
        return F.cross_entropy(model(Xtr), ytr).item(), (model(Xte).argmax(1) == yte).float().mean().item()

results = {}
t = time.time()
for n_blocks in [2, 10, 25]:
    for residual in [False, True]:
        torch.manual_seed(0)
        results[(2 * n_blocks + 2, residual)] = train_eval(Net(n_blocks, residual))
print(f"done in {time.time() - t:.0f} s\n")
print(f"{'layers':>6} | {'plain: train loss / test acc':>30} | {'residual: train loss / test acc':>32}")
for L in [6, 22, 52]:
    (lp, ap), (lr_, ar) = results[(L, False)], results[(L, True)]
    print(f"{L:>6} | {lp:>15.3f} / {ap:>9.1%} | {lr_:>17.3f} / {ar:>9.1%}")

done in 214 s

layers |   plain: train loss / test acc |  residual: train loss / test acc
     6 |           0.687 /     78.1% |             0.291 /     88.0%
    22 |           2.303 /     10.2% |             0.587 /     76.1%
    52 |           2.303 /     10.2% |             0.816 /     66.1%


- At **6 layers** both train, and the residual one is already better.
- At **22 and 52 layers** the plain network is **stuck at 10%**, which is random guessing among 10 digits. Its training loss stays at 2.30 = ln(10), the loss of a model that knows nothing. **It cannot even fit its training data.** This is the *degradation problem* the ResNet paper described.
- The **residual** networks of the same depth train fine.

Our deep residual nets score lower than the 6-layer one only because they need more epochs and BatchNorm. Real ResNets add BatchNorm after every convolution, and that plus skip connections let them train with **50, 101, even 152 layers**.

> **Exam answer (Hard Q4):** In a plain deep network the gradient reaching early layers is a product of many per-layer factors, so it shrinks toward zero (vanishing gradients) and adding layers makes even *training* accuracy worse (degradation). A residual block outputs F(x) + x, so the gradient is F′(x) + 1. The identity path carries the gradient back unchanged, and a block that isn't useful can learn F(x) ≈ 0 and pass its input through. Extra depth therefore never hurts, and networks with 100+ layers become trainable.

### Real ResNets

In [4]:
from torchvision.models import resnet18, resnet34, resnet50, resnet101, resnet152

print(f"{'model':<10}{'weight layers':>15}{'parameters':>14}")
for fn in [resnet18, resnet34, resnet50, resnet101, resnet152]:
    net = fn(weights=None)
    layers = sum(isinstance(m, (nn.Conv2d, nn.Linear)) for m in net.modules())
    downsample = sum(isinstance(m, nn.Conv2d) and m.kernel_size == (1, 1) and "downsample" in name
                     for name, m in net.named_modules())
    params = sum(p.numel() for p in net.parameters())
    print(f"{fn.__name__:<10}{layers - downsample:>15}{params / 1e6:>12.1f} M")
    del net

model       weight layers    parameters


resnet18               18        11.7 M


resnet34               34        21.8 M


resnet50               50        25.6 M


resnet101             101        44.5 M


resnet152             152        60.2 M


The number in the name is the count of weight layers on the main path. The four 1×1 "downsample" convolutions on skip connections, which match sizes when the shape changes, are not counted. **ResNet-152 has 152 layers but fewer parameters (60 M) than VGG-16 (138 M)**, thanks to "bottleneck" blocks (1×1 → 3×3 → 1×1 convs) and **no giant fully connected layers**, just global average pooling and one FC layer.

**Achievement:** ResNet won ImageNet 2015 with **3.57% top-5 error**, better than the estimated human error of about 5%.
**Real-world use (Medium Q8):** ResNet is the default backbone in **medical imaging**, e.g. classifying chest X-rays or skin lesions, often **pretrained on ImageNet and fine-tuned** on a smaller medical dataset. It's also the backbone inside Faster R-CNN and Mask R-CNN (notebooks 04–05).

### Skip connections in ResNet vs U-Net (preview of Hard Q7)
| | ResNet | U-Net (notebook 05) |
|---|---|---|
| Connects | a block's input to its output (short jumps) | encoder layer → matching decoder layer (long jumps) |
| Combines by | **adding** (F(x) + x) | **concatenating** feature maps |
| Main purpose | **gradient flow**: train very deep networks | **restore fine spatial detail** lost by downsampling |

## 2 · EfficientNet (2019): scaling the smart way

**Problem solved:** to get a more accurate CNN, people scaled it up in **one** direction:
- **depth**: more layers (ResNet-18 → ResNet-152),
- **width**: more channels/filters per layer,
- **resolution**: bigger input images (224 → 600 pixels).

Scaling just one of these quickly hits **diminishing returns**: lots more computation for a tiny accuracy gain. It's like a bigger bike with only a bigger front wheel.

**Innovation: compound scaling.** Scale **all three together** with one formula, controlled by a single number φ ("phi", how much more compute you can afford):

  depth × **α^φ**,  width × **β^φ**,  resolution × **γ^φ**,  with **α · β² · γ² ≈ 2**

EfficientNet uses **α = 1.2, β = 1.1, γ = 1.15**. Each +1 in φ then roughly **doubles the computation** (FLOPs). Width and resolution are squared because doubling either one makes the work per layer about 4× bigger.

In [5]:
alpha, beta, gamma = 1.2, 1.1, 1.15
print(f"alpha * beta^2 * gamma^2 = {alpha * beta**2 * gamma**2:.3f}  (~2 -> each step doubles compute)\n")
print(f"{'phi':>3} | {'depth x':>8} | {'width x':>8} | {'resolution':>10} | {'compute x':>9}")
for phi in range(0, 8):
    d, w, r = alpha ** phi, beta ** phi, gamma ** phi
    print(f"{phi:>3} | {d:>8.2f} | {w:>8.2f} | {224 * r:>8.0f}px | {d * w**2 * r**2:>9.1f}")

alpha * beta^2 * gamma^2 = 1.920  (~2 -> each step doubles compute)

phi |  depth x |  width x | resolution | compute x
  0 |     1.00 |     1.00 |      224px |       1.0
  1 |     1.20 |     1.10 |      258px |       1.9
  2 |     1.44 |     1.21 |      296px |       3.7
  3 |     1.73 |     1.33 |      341px |       7.1
  4 |     2.07 |     1.46 |      392px |      13.6
  5 |     2.49 |     1.61 |      451px |      26.1
  6 |     2.99 |     1.77 |      518px |      50.1
  7 |     3.58 |     1.95 |      596px |      96.3


Every step grows depth, width *and* resolution a bit. The released models **B0 (φ = 0) to B7** follow this recipe, with input sizes from 224 up to 600 pixels.

**Why is this better than just going deeper or wider? (Medium Q7, Hard Q3)**
The three dimensions **depend on each other**. A higher-resolution image has more detail, so the network needs **more layers** (a larger receptive field to cover bigger objects) **and more channels** (to capture the extra fine-grained patterns). Growing only one dimension leaves the others as the bottleneck. Balanced growth uses each extra FLOP where it helps most. The constants α, β and γ were found by a small grid search on the base model, then reused for all sizes.

### The payoff: accuracy per parameter

In [6]:
from torchvision.models import efficientnet_b0, efficientnet_b7, resnet50, mobilenet_v3_large

def count(fn):
    net = fn(weights=None)
    n = sum(p.numel() for p in net.parameters())
    del net
    return n / 1e6

# ImageNet top-1 accuracies as reported in the papers (we can't train these here)
table = [("ResNet-50", count(resnet50), 76.0),
         ("EfficientNet-B0", count(efficientnet_b0), 77.1),
         ("EfficientNet-B7", count(efficientnet_b7), 84.3),
         ("MobileNetV3-Large", count(mobilenet_v3_large), 75.2)]
print(f"{'model':<20}{'parameters':>12}{'ImageNet top-1':>16}")
for name, p, acc in table:
    print(f"{name:<20}{p:>10.1f} M{acc:>15.1f}%")

model                 parameters  ImageNet top-1
ResNet-50                 25.6 M           76.0%
EfficientNet-B0            5.3 M           77.1%
EfficientNet-B7           66.3 M           84.3%
MobileNetV3-Large          5.5 M           75.2%


**EfficientNet-B0 beats ResNet-50 with about 5× fewer parameters.** In the paper, EfficientNet-B7 matched the best model of the time (84.3%) while being **8.4× smaller** and much faster.

**Real-world use (Medium Q9):** on-device **mobile apps**, e.g. a phone app that recognises plant diseases or food from the camera, and anything else needing good accuracy on limited hardware (drones, embedded cameras).

## 3 · Lightweight models for phones: MobileNet (Easy Q14)

The course's answer for "lightweight yet accurate for mobile vision apps" is **EfficientNet**. Its smallest versions (B0) are tiny. Its building block comes from **MobileNet**, the other classic mobile architecture, which uses **depthwise-separable convolutions**:

- **Normal conv:** every filter looks at all input channels at once, giving K × K × C_in × C_out weights.
- **Depthwise-separable:** ① one K×K filter **per channel** (depthwise), then ② a **1×1 conv** to mix the channels (pointwise), giving K × K × C_in + C_in × C_out weights.

In [7]:
C_in, C_out, K = 128, 128, 3
normal = nn.Conv2d(C_in, C_out, K, padding=1, bias=False)
separable = nn.Sequential(nn.Conv2d(C_in, C_in, K, padding=1, groups=C_in, bias=False),   # depthwise
                          nn.Conv2d(C_in, C_out, 1, bias=False))                          # pointwise 1x1
n1 = sum(p.numel() for p in normal.parameters())
n2 = sum(p.numel() for p in separable.parameters())
x = torch.randn(1, C_in, 56, 56)
print("same output shape:", normal(x).shape == separable(x).shape)
print(f"normal 3x3 conv     : {n1:>7,} weights")
print(f"depthwise-separable : {n2:>7,} weights  -> {n1 / n2:.1f}x fewer (and about that much less computation)")

same output shape: True
normal 3x3 conv     : 147,456 weights
depthwise-separable :  17,536 weights  -> 8.4x fewer (and about that much less computation)


---
## Key takeaways (Key Concepts sheet)
- **ResNet:** made very deep networks trainable. **Learn residuals instead of direct mappings**; skip connections let gradients flow. 50/101/152 layers, state of the art on ImageNet 2015. Used in **medical imaging**.
- **EfficientNet:** made scaling efficient and practical. **Compound scaling** of depth, width and resolution with one formula: high accuracy with fewer parameters. Used in **mobile vision apps**.
- Together they represent the **modern era of CNNs** (solving depth and efficiency), and their ideas (skip connections, careful scaling) carried straight into **Vision Transformers**, which use residual connections in every block (notebook 07).

## ✅ Check yourself
1. What problem does ResNet solve? *(vanishing gradients / degradation when training very deep networks)*
2. ResNet's key innovation? *(residual (skip) connections: output = F(x) + x)*
3. Direct mapping vs residual? *(a plain block learns H(x) itself; a residual block learns only the change F(x) = H(x) − x)*
4. Why does the gradient survive in ResNet? *(d/dx [F(x) + x] = F′(x) + 1; the +1 path carries it backwards)*
5. Which architecture made 152-layer networks trainable? *(ResNet)*
6. What does EfficientNet's compound scaling balance? *(depth, width and resolution, all together via φ)*
7. Lightweight but accurate for phones? *(EfficientNet (B0) / MobileNet)*